In [2]:
from pypot.dynamixel import DxlIO
from pypot.dynamixel.protocol.v1 import *

from glob import glob

ports = glob('/dev/ttyACM*')
assert len(ports) == 1

port = ports[0]
print('Connecting on port: {}'.format(port))
dxl_io = DxlIO(port)

poulpe_id = 42
N_AXIS = 2

Connecting on port: /dev/ttyACM1


In [3]:
ping_packet = DxlPingPacket(poulpe_id)
dxl_io._send_packet(ping_packet)

DxlStatusPacket(id=42, error=0, parameters=())

In [4]:
import struct

def read_current_pos():
    pos_packet = DxlReadDataPacket(poulpe_id, 50, N_AXIS * 4)
    resp = dxl_io._send_packet(pos_packet)
    data = bytearray(resp.parameters)
    pos = struct.unpack(N_AXIS * 'f', data)
    return pos

read_current_pos()

(0.0, 0.0)

In [5]:
import struct

def read_target_position():
    pos_packet = DxlReadDataPacket(poulpe_id, 60, N_AXIS * 4)
    resp = dxl_io._send_packet(pos_packet)
    data = bytearray(resp.parameters)
    pos = struct.unpack(N_AXIS * 'f', data)
    return pos

read_target_position()

(0.0, 0.0)

In [6]:
def write_target_position(target):
    p = DxlWriteDataPacket(poulpe_id, 60, struct.pack(N_AXIS * 'f', *target))
    resp = dxl_io._send_packet(p)
    return resp

write_target_position([0.0,0.0,])

DxlStatusPacket(id=42, error=0, parameters=())

In [11]:
def read_torque_enabled():
    p = DxlReadDataPacket(poulpe_id, 40, N_AXIS)
    resp = dxl_io._send_packet(p)
    data = bytearray(resp.parameters)
    torque = struct.unpack(N_AXIS * '?', data)
    return torque

read_torque_enabled()

(False, False)

In [12]:
def write_torque_enabled(torque):
    p = DxlWriteDataPacket(poulpe_id, 40, struct.pack(N_AXIS * '?', *torque))
    resp = dxl_io._send_packet(p)
    return resp

write_torque_enabled([True,True,])

DxlStatusPacket(id=42, error=0, parameters=())

In [15]:
import time
import numpy as np

pos = []
send_target = []
read_target = []

t0 = time.time()
while True:
    if time.time() - t0 > 10:
        break

    target = [
        #np.random.rand(), 
        90 * np.sin(2 * np.pi * 0.5 * time.time()),
        90 * np.sin(2 * np.pi * 0.5 * time.time()),
    ]
    write_target_position(target)
    send_target.append(target)
    time.sleep(0.001)
    pos.append(read_current_pos())
    time.sleep(0.001)
    read_target.append(read_target_position())
    time.sleep(0.001)

In [16]:
np.array(send_target) - np.array(read_target)

array([[-1.10607759e-07,  3.39640582e-09],
       [ 2.02047795e-07, -1.43975684e-07],
       [-4.30988845e-08, -5.90103513e-08],
       ...,
       [ 2.59350016e-08,  1.43954176e-08],
       [-3.10575232e-08, -3.36657602e-08],
       [ 8.02282774e-09, -6.60959083e-08]])

In [36]:
write_torque_enabled([False,False,])

DxlStatusPacket(id=42, error=0, parameters=())